# Datacube subsetting

A common task in glaciology is the extraction (and subsequent analysis) of velocity fields for a single glacier basin or region over a period of time. For this type of analysis, where we need a reasonable amount of information in both the spatial and temporal dimensions, it is typically best to use the `cubed` Zarr stores. The relevant Zarr for Greenland is called `greenland_multisource_velocity_timeseries.zarr` whilst the Antarctic equivalent is called `antarctica_multisource_velocity_timeseries.zarr` and their equivalent URLs are:  

In [1]:
# Greenland URL
greenland_url = "https://data.source.coop/uos-shiver/greenland/greenland_multisource_velocity_cubed.zarr"

# Antarctica URL
antarctica_url = "https://data.source.coop/uos-shiver/antarctica/antarctica_multisource_velocity_cubed.zarr"


The example below demonstrates how to extract a 3D subset over Sermeq Kujalleq for the 2021 calendar year. Because the data is loaded lazily using Dask, this spatial and temporal crop is fast and requires no download. Of course, if we wanted to go further and perform operations on the subset or export it as a NetCDF, we would need to download it or at least stream it to NetCDF. 


In [2]:
import xarray as xr
from pyproj import Transformer

# 1. Load the cubed Zarr store
url = "https://data.source.coop/uos-shiver/antarctica/antarctica_multisource_velocity_cubed.zarr"
ds_cube = xr.open_zarr(url, consolidated=True, chunks={})

# 2. Convert geographic coordinates (Lon/Lat) to EPSG:3031 (X/Y)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
lon_top_left, lat_top_left = -95.002, -74.807
lon_bot_right, lat_bot_right = -104.606, -75.345
xmin, ymax = transformer.transform(lon_top_left, lat_top_left)
xmax, ymin = transformer.transform(lon_bot_right, lat_bot_right)

# Handle Y-axis orientation for slicing
y_slice = slice(ymax, ymin) if ds_cube.y[0] > ds_cube.y[-1] else slice(ymin, ymax)

# 3. Subset spatially and temporally (year 2021)
subset = ds_cube.sortby("time").sel(
    x=slice(xmin, xmax),
    y=y_slice,
    time=slice("2021-01-01", "2021-12-31")
)

# 4. Select specific variables to isolate the data you need
vars_to_keep = ["speed", "vx", "vy", "speed_error", "vx_error", "vy_error", "data_source"]
subset = subset[vars_to_keep]

# 5. Display the dataset structure
subset

<xarray.Dataset> Size: 5GB
Dimensions:      (time: 282, y: 1294, x: 524)
Coordinates:
  * time         (time) datetime64[ns] 2kB 2021-01-01T00:00:00.504000 ... 202...
  * y            (y) float64 10kB -1.449e+05 -1.451e+05 ... -4.035e+05
  * x            (x) float64 4kB -1.654e+06 -1.654e+06 ... -1.549e+06 -1.549e+06
Data variables:
    speed        (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    vx           (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    vy           (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    speed_error  (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    vx_error     (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    vy_error     (time, y, x) float32 765MB dask.array<chunksize=(136, 42, 73), meta=np.ndarray>
    data_source  (time) <U50 56kB dask.array<chunksize=(282,), meta=np.ndarray>